# E-Commerce Customer Insights Engine

**Week 2 milestone project | Python, Pandas, Jupyter Notebook**

This notebook joins customer and order CSV files, calculates customer and city-level KPIs, identifies non-converting customers, and produces a Top 5 spending leaderboard.

**Dataset note:** The included CSV files are sample data created for demonstration. Replace them with the internship-provided files if those were supplied separately.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
customers = pd.read_csv("customers.csv")
orders = pd.read_csv("orders.csv")

print("Customers dataset:")
display(customers)
print("Orders dataset:")
display(orders)

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 1, in <module>
  File "/opt/pyvenv/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
  File "/opt/pyvenv/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
  File "/opt/pyvenv/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^
  File "/opt/pyvenv/lib/python3.13/site-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
                   ~~~~~~~~~~^
        f,
        ^^
    ...<6 lines>...
        storage_options=

In [3]:
# Inner join: only matching customer/order records
inner_join = pd.merge(customers, orders, on="customer_id", how="inner")
print("Inner join:")
display(inner_join)

# Left join: retain every customer, including customers with no orders
left_join = pd.merge(customers, orders, on="customer_id", how="left")
print("Left join:")
display(left_join)

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 2, in <module>
NameError: name 'customers' is not defined


In [4]:
# Customer-level metrics
customer_spending = (
    inner_join.groupby(["customer_id", "customer_name", "city"], as_index=False)
    .agg(
        total_spend=("order_amount", "sum"),
        total_orders=("order_id", "count"),
        average_order_value=("order_amount", "mean")
    )
)
customer_spending["average_order_value"] = customer_spending["average_order_value"].round(2)
print("Customer spending metrics:")
display(customer_spending.sort_values("total_spend", ascending=False))

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 3, in <module>
NameError: name 'inner_join' is not defined


In [5]:
# City/region business metrics
city_analysis = (
    inner_join.groupby(["city"], as_index=False)
    .agg(
        total_sales=("order_amount", "sum"),
        total_orders=("order_id", "count"),
        unique_customers=("customer_id", "nunique")
    )
)
print("City-wise sales:")
display(city_analysis.sort_values("total_sales", ascending=False))

region_analysis = (
    inner_join.merge(customers[["customer_id", "region"]], on="customer_id", how="left")
    .groupby("region", as_index=False)
    .agg(total_sales=("order_amount", "sum"), total_orders=("order_id", "count"))
)
print("Region-wise sales:")
display(region_analysis)

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 3, in <module>
NameError: name 'inner_join' is not defined


In [6]:
# Find customers with zero purchases, and prepare a clean customer summary
customer_summary = customers.merge(
    orders.groupby("customer_id", as_index=False).agg(
        total_spend=("order_amount", "sum"),
        total_orders=("order_id", "count")
    ),
    on="customer_id",
    how="left"
)
customer_summary[["total_spend", "total_orders"]] = customer_summary[["total_spend", "total_orders"]].fillna(0)
customer_summary["total_spend"] = customer_summary["total_spend"].astype(int)
customer_summary["total_orders"] = customer_summary["total_orders"].astype(int)

zero_purchase_customers = customer_summary[customer_summary["total_orders"] == 0]
print("Customers with zero purchases:")
display(zero_purchase_customers if not zero_purchase_customers.empty else pd.DataFrame(columns=customer_summary.columns))

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 2, in <module>
NameError: name 'customers' is not defined


In [7]:
# Top 5 highest-spending customers
top_5_customers = (
    customer_summary[customer_summary["total_orders"] > 0]
    .sort_values(["total_spend", "customer_id"], ascending=[False, True])
    .head(5)
)
print("TOP 5 HIGHEST-SPENDING CUSTOMERS:")
display(top_5_customers)

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 3, in <module>
NameError: name 'customer_summary' is not defined


In [8]:
# Final business summary
print("===== E-COMMERCE CUSTOMER INSIGHTS =====")
print("Total customers:", len(customers))
print("Total orders:", len(orders))
print("Total revenue:", int(orders["order_amount"].sum()))
print("Average order value (AOV):", round(orders["order_amount"].mean(), 2))
print("Unique purchasing customers:", orders["customer_id"].nunique())

# Save key result tables for easy sharing
customer_spending.to_csv("customer_spending_summary.csv", index=False)
city_analysis.to_csv("city_sales_summary.csv", index=False)
top_5_customers.to_csv("top_5_customers.csv", index=False)
print("Saved: customer_spending_summary.csv, city_sales_summary.csv, top_5_customers.csv")

Traceback (most recent call last):
  File "/tmp/ipykernel_249/956011701.py", line 188, in <module>
    exec(compile(code, "<notebook-cell>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<notebook-cell>", line 3, in <module>
NameError: name 'customers' is not defined
